# BerryNet — Full Pipeline: Balanced Training + Transfer Learning + Grad-CAM + Severity + Continuous Learning

This notebook builds on your fixed BerryNet pipeline and adds, in order:

1. **A visible class-balance / bias report** after training — not just overall accuracy, but per-class precision/recall/F1 with an automatic flag if one class is lagging.
2. **A transfer-learning model option** (EfficientNet-B0 pretrained on ImageNet) alongside your original custom BerryNet — this is the single biggest lever for pushing accuracy past 90% on a small-ish dataset. You can switch between the two with one variable.
3. **Grad-CAM** — visualizes what the model looked at.
4. **Severity scoring** — HSV-based lesion segmentation → Mild / Moderate / Severe.
5. **A continuous learning demonstration** — done honestly: a simulated two-phase data stream (Phase A: 2 classes, Phase B: introduces the 3rd class), comparing naive fine-tuning (which forgets) against a replay-buffer approach (which doesn't), with real before/after numbers. This is clearly a *demonstration* of the concept on your dataset, not a production continual-learning system — I'll explain why that distinction matters at that section.

**Honest expectations set up front:** I can't guarantee >90% accuracy — that depends on your actual images, not the code. What I *can* guarantee is that this notebook gives you the best realistic shot at it (transfer learning) and tells you clearly, via the bias report and accuracy check, whether what you got is good enough or needs another pass.

## 0. Get the dataset from Kaggle

In [1]:
from google.colab import files
uploaded = files.upload()  # select your kaggle.json here

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
!pip install -q kaggle


Saving kaggle.json to kaggle.json


In [2]:
!kaggle datasets download -d nahiduzzaman13/mulberry-leaf-dataset


Dataset URL: https://www.kaggle.com/datasets/nahiduzzaman13/mulberry-leaf-dataset
License(s): CC0-1.0
100% 10.7G/10.7G [09:37<00:00, 19.8MB/s]



In [3]:
import zipfile
import os

zip_file = "mulberry-leaf-dataset.zip"
extract_path = "/content/dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")


Dataset extracted successfully!


In [4]:
for root, dirs, fs in os.walk(extract_path):
    depth = root.replace(extract_path, "").count(os.sep)
    if depth <= 2:
        print(root, "->", dirs)


/content/dataset -> ['Mulberry Data']
/content/dataset/Mulberry Data -> ['Leaf spot', 'Disease Free leaves', 'Leaf Rust']
/content/dataset/Mulberry Data/Leaf spot -> []
/content/dataset/Mulberry Data/Disease Free leaves -> []
/content/dataset/Mulberry Data/Leaf Rust -> []


Set `DATASET_PATH` below once you see the real folder structure printed above.

In [5]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from tqdm import tqdm

from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.model_selection import train_test_split
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


## 1. Data

In [6]:
DATASET_PATH = "/content/dataset/Mulberry Data"  # adjust based on the folder walk above

base_dataset = ImageFolder(root=DATASET_PATH)

print("Total Images  :", len(base_dataset))
print("Classes       :", base_dataset.classes)
print("Class Mapping :", base_dataset.class_to_idx)

counts = Counter(base_dataset.targets)
print("\nPer-class image counts:")
for idx, name in enumerate(base_dataset.classes):
    print(f"  {name:22s}: {counts[idx]}")


Total Images  : 1091
Classes       : ['Disease Free leaves', 'Leaf Rust', 'Leaf spot']
Class Mapping : {'Disease Free leaves': 0, 'Leaf Rust': 1, 'Leaf spot': 2}

Per-class image counts:
  Disease Free leaves   : 440
  Leaf Rust             : 489
  Leaf spot             : 162


In [7]:
labels = base_dataset.targets
indices = np.arange(len(base_dataset))

train_val_idx, test_idx = train_test_split(
    indices, test_size=0.10, stratify=labels, random_state=SEED
)

train_val_labels = [labels[i] for i in train_val_idx]
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.20, stratify=train_val_labels, random_state=SEED
)

print("Train      :", len(train_idx))
print("Validation :", len(val_idx))
print("Test       :", len(test_idx))


Train      : 784
Validation : 197
Test       : 110


In [8]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_base = ImageFolder(root=DATASET_PATH, transform=train_transform)
eval_base  = ImageFolder(root=DATASET_PATH, transform=test_transform)

train_dataset = Subset(train_base, train_idx)
val_dataset   = Subset(eval_base, val_idx)
test_dataset  = Subset(eval_base, test_idx)

class_names = base_dataset.classes
num_classes = len(class_names)

print("Train Images     :", len(train_dataset))
print("Validation Images:", len(val_dataset))
print("Test Images      :", len(test_dataset))


Train Images     : 784
Validation Images: 197
Test Images      : 110


## 2. Balance the batches (class-imbalance fix, unchanged from your original)

In [9]:
BATCH_SIZE = 32

train_labels_only = [base_dataset.targets[i] for i in train_idx]
class_counts = Counter(train_labels_only)

print("Training set class counts:", class_counts)

class_weights = {c: len(train_labels_only) / (num_classes * n) for c, n in class_counts.items()}
print("Per-class weights:", class_weights)

sample_weights = [class_weights[label] for label in train_labels_only]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)


Training set class counts: Counter({1: 351, 0: 316, 2: 117})
Per-class weights: {1: 0.7445394112060779, 0: 0.8270042194092827, 2: 2.2336182336182335}


## 3. Two model options

`MODEL_TYPE = "custom"` uses your original BerryNet, trained from scratch.
`MODEL_TYPE = "transfer"` uses EfficientNet-B0 pretrained on ImageNet with a
new 3-class head — pretrained features are the most reliable way to boost
accuracy on a dataset this size, since the backbone already knows general
edge/texture/color features and only needs to learn what's specific to leaf
disease.

**Recommendation: start with `"transfer"` if hitting 90%+ matters to you.**
Keep `"custom"` available since it's your own designed architecture and may
matter more for your report's originality story — you can run both and
compare, the notebook is written so either just works.

In [10]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride,
                      padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class NobelBlock(nn.Module):
    """Inception-style multi-branch block."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.branch1 = ConvBlock(in_channels, out_channels, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBlock(in_channels, out_channels, kernel_size=1),
            ConvBlock(out_channels, out_channels, kernel_size=3, padding=1),
        )
        self.branch3 = nn.Sequential(
            ConvBlock(in_channels, out_channels, kernel_size=1),
            ConvBlock(out_channels, out_channels, kernel_size=5, padding=2),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            ConvBlock(in_channels, out_channels, kernel_size=1),
        )

    def forward(self, x):
        b1, b2, b3, b4 = self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.conv2(out)
        out += identity
        return self.relu(out)


class SqueezeBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x):
        out = self.block(x)
        return torch.cat([x, out], dim=1)


class DenseBlock(nn.Module):
    def __init__(self, in_channels, growth_rate, num_layers):
        super().__init__()
        layers = []
        channels = in_channels
        for _ in range(num_layers):
            layers.append(DenseLayer(channels, growth_rate))
            channels += growth_rate
        self.block = nn.Sequential(*layers)
        self.out_channels = channels

    def forward(self, x):
        return self.block(x)


class BerryNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBlock(3, 32, kernel_size=3, padding=1),
            ConvBlock(32, 64, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.nobel = NobelBlock(in_channels=64, out_channels=32)
        self.residual = ResidualBlock(in_channels=128, out_channels=128)
        self.squeeze = SqueezeBlock(channels=128)
        self.dense = DenseBlock(in_channels=128, growth_rate=32, num_layers=4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Linear(self.dense.out_channels, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.nobel(x)
        x = self.residual(x)
        x = self.squeeze(x)
        x = self.dense(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.classifier(x)


def build_model(model_type, num_classes=3):
    if model_type == "custom":
        return BerryNet(num_classes=num_classes)
    elif model_type == "transfer":
        weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
        m = torchvision.models.efficientnet_b0(weights=weights)
        in_features = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_features, num_classes)
        return m
    else:
        raise ValueError("model_type must be 'custom' or 'transfer'")


def get_gradcam_target_layer(model, model_type):
    """Last spatial feature map before global pooling — Grad-CAM's hook point."""
    if model_type == "custom":
        return model.dense
    elif model_type == "transfer":
        return model.features[-1]
    else:
        raise ValueError("model_type must be 'custom' or 'transfer'")


# sanity check for the custom model
_x = torch.randn(2, 3, 224, 224)
print("Custom BerryNet output shape :", build_model("custom")(_x).shape)


Custom BerryNet output shape : torch.Size([2, 3])


## 3.1 Adaptive Synergistic Loss Function (ASLF) — reproduced from Nobel et al. (2026)

**Attribution, read this before using this cell in any report:** the loss
function below is a reproduction of the *Adaptive Synergistic Loss Function*
(ASLF) proposed in:

> Nobel, S.M.N., Tasir, M.A.M., Sultana, S., Al-Moisheer, A.S., Moni, M.A.
> "An explainable deep learning model for mulberry leaf classification and
> disease detection." *Engineering Applications of Artificial Intelligence*,
> 165 (2026), 113470.

ASLF is **their published contribution, not an original method developed
here**. It is included so you can reproduce their reported approach and
compare it against plain cross-entropy on your own data/split — this is a
legitimate thing to do (a reproduction/comparison), but it must be cited as
theirs in any write-up. Do not present ASLF as your own novel loss function.

ASLF combines four things in one loss term:
1. **Class-balanced weighting** (Cui et al.-style effective-number weighting, controlled by β)
2. **Label smoothing** (controlled by α) to reduce overconfidence
3. **Focal-style hard-example emphasis** (controlled by γ) to focus on
   currently-misclassified examples
4. **Feature-compactness regularization** (controlled by λ) that pulls each
   class's feature vectors toward their own class centroid, encouraging
   tighter, more separable clusters in the penultimate layer

The paper's reported optimal hyperparameters are α=0.1, β=0.999, γ=2, λ=0.5 —
used as the defaults below.

In [11]:
class FeatureHook:
    """
    Captures the input to a given module, such as the classifier head.

    The captured tensor is the penultimate-layer feature vector used
    by the compactness term of ASLF.
    """

    def __init__(self, module):
        self.features = None
        self.hook = module.register_forward_hook(self._hook)

    def _hook(self, module, input, output):
        self.features = input[0]

    def remove(self):
        self.hook.remove()


class AdaptiveSynergisticLoss(nn.Module):
    """
    Adaptive Synergistic Loss Function (ASLF).

    Combines:
    1. Class-balanced weighting
    2. Label smoothing
    3. Focal-style modulation
    4. Feature compactness regularization
    """

    def __init__(
        self,
        num_classes,
        class_counts=None,
        alpha=0.1,
        beta=0.999,
        gamma=2.0,
        lam=0.5
    ):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma
        self.lam = lam

        # ------------------------------------------
        # Class-balanced weights
        # ------------------------------------------

        if class_counts is not None:

            counts = torch.tensor(
                [
                    max(class_counts.get(c, 0), 1)
                    for c in range(num_classes)
                ],
                dtype=torch.float32
            )

            w = (1 - beta) / (1 - beta ** counts)

            w = w / w.sum() * num_classes

        else:
            w = torch.ones(num_classes)

        self.register_buffer(
            "class_weights",
            w
        )

    def forward(self, logits, targets, features=None):

        device = logits.device

        w = self.class_weights.to(device)

        n, c = logits.shape

        # ------------------------------------------
        # Label smoothing
        # ------------------------------------------

        with torch.no_grad():

            y_sm = torch.full(
                (n, c),
                self.alpha / (c - 1),
                device=device
            )

            y_sm.scatter_(
                1,
                targets.unsqueeze(1),
                1.0 - self.alpha
            )

        # ------------------------------------------
        # Log probabilities
        # ------------------------------------------

        log_p = torch.log_softmax(
            logits,
            dim=1
        )

        p = log_p.exp()

        # ------------------------------------------
        # Class-weighted cross entropy
        # ------------------------------------------

        ce_i = -(
            w.unsqueeze(0)
            * y_sm
            * log_p
        ).sum(dim=1)

        # ------------------------------------------
        # Focal-style modulation
        # ------------------------------------------

        if self.gamma > 0:

            p_t = (
                y_sm * p
            ).sum(dim=1)

            ce_i = (
                (1 - p_t) ** self.gamma
            ) * ce_i

        ce = ce_i.mean()

        # ------------------------------------------
        # Feature compactness term
        # ------------------------------------------

        comp = torch.tensor(
            0.0,
            device=device
        )

        if self.lam > 0 and features is not None:

            comp_terms = []

            for cls in targets.unique():

                mask = targets == cls

                # Mean feature vector for this class
                mu_c = features[mask].mean(
                    dim=0,
                    keepdim=True
                )

                # Squared distance from class mean
                d = (
                    (features[mask] - mu_c) ** 2
                ).sum(dim=1)

                comp_terms.append(d)

            if comp_terms:
                comp = torch.cat(
                    comp_terms
                ).mean()

        # ------------------------------------------
        # Final ASLF
        # ------------------------------------------

        return ce + self.lam * comp

In [12]:
MODEL_TYPE = "transfer"   # "custom" or "transfer"
LOSS_TYPE = "aslf"        # "cross_entropy" or "aslf"


# ==========================================
# Build model
# ==========================================

model = build_model(
    MODEL_TYPE,
    num_classes=num_classes
).to(device)


# ==========================================
# Transfer-learning setup
# ==========================================

if MODEL_TYPE == "transfer":

    # Freeze all parameters
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze classifier
    for param in model.classifier.parameters():
        param.requires_grad = True

    # Unfreeze last 3 feature blocks
    for block in list(model.features.children())[-3:]:
        for param in block.parameters():
            param.requires_grad = True

    # Count trainable parameters
    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    print(
        f"Trainable params: {trainable:,} / "
        f"{total:,} "
        f"({100 * trainable / total:.1f}%)"
    )


# ==========================================
# Feature hook for ASLF
# ==========================================

feature_hook = FeatureHook(
    model.classifier
)


# ==========================================
# Loss function
# ==========================================

if LOSS_TYPE == "aslf":

    criterion = AdaptiveSynergisticLoss(
        num_classes=num_classes,
        class_counts=class_counts
    ).to(device)

else:

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )


# ==========================================
# Optimizer
# ==========================================

optimizer = optim.Adam(
    [
        p for p in model.parameters()
        if p.requires_grad
    ],
    lr=1e-3 if MODEL_TYPE == "custom" else 3e-4,
    weight_decay=1e-4,
)


# ==========================================
# Learning-rate scheduler
# ==========================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=3
)


print(
    f"Training model_type = '{MODEL_TYPE}', "
    f"loss_type = '{LOSS_TYPE}'"
)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 158MB/s]


Trainable params: 3,159,583 / 4,011,391 (78.8%)
Training model_type = 'transfer', loss_type = 'aslf'


## 4. Train / validate

In [13]:
from sklearn.metrics import f1_score


def compute_loss(outputs, labels, criterion):
    """
    Compute the selected loss.

    ASLF requires the penultimate feature vector captured
    by feature_hook.
    """

    if LOSS_TYPE == "aslf":
        return criterion(
            outputs,
            labels,
            features=feature_hook.features
        )

    return criterion(outputs, labels)


# =========================================================
# Train for one epoch
# =========================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in tqdm(
        loader,
        desc="Training",
        leave=False
    ):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = compute_loss(
            outputs,
            labels,
            criterion
        )

        # Backpropagation
        loss.backward()

        optimizer.step()

        # Statistics
        running_loss += loss.item()

        preds = torch.argmax(
            outputs,
            dim=1
        )

        running_correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / len(loader)

    epoch_acc = (
        100.0 * running_correct / total
    )

    return epoch_loss, epoch_acc


# =========================================================
# Validation
# =========================================================

def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    running_correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Calculate loss
            loss = compute_loss(
                outputs,
                labels,
                criterion
            )

            running_loss += loss.item()

            preds = torch.argmax(
                outputs,
                dim=1
            )

            running_correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    val_loss = (
        running_loss / len(loader)
    )

    val_acc = (
        100.0 * running_correct / total
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return (
        val_loss,
        val_acc,
        val_macro_f1
    )


# =========================================================
# Per-class evaluation
# =========================================================

def evaluate_per_class(
    model,
    loader,
    class_names,
    device
):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            preds = torch.argmax(
                outputs,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # ---------------------------------------------
    # Per-class accuracy
    # ---------------------------------------------

    per_class_acc = {}

    for idx, name in enumerate(class_names):

        mask = all_labels == idx

        if mask.sum() > 0:

            per_class_acc[name] = (
                100.0
                * (all_preds[mask] == idx).sum()
                / mask.sum()
            )

        else:

            per_class_acc[name] = None

    # ---------------------------------------------
    # Overall accuracy
    # ---------------------------------------------

    overall_acc = (
        100.0
        * (all_preds == all_labels).sum()
        / len(all_labels)
    )

    # ---------------------------------------------
    # Macro F1
    # ---------------------------------------------

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return (
        per_class_acc,
        overall_acc,
        macro_f1
    )

In [ ]:
EPOCHS = 30
EARLY_STOP_PATIENCE = 7

best_val_f1 = -1.0
epochs_without_improvement = 0

CHECKPOINT_PATH = f"BerryNet_best_{MODEL_TYPE}.pth"

train_losses = []
val_losses = []

train_accs = []
val_accs = []


# =========================================================
# Training loop
# =========================================================

for epoch in range(EPOCHS):

    print("=" * 60)
    print(f"Epoch [{epoch + 1}/{EPOCHS}]")
    print("=" * 60)

    # ---------------------------------------------
    # Training
    # ---------------------------------------------

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # ---------------------------------------------
    # Validation
    # ---------------------------------------------

    val_loss, val_acc, val_macro_f1 = validate(
        model,
        val_loader,
        criterion,
        device
    )

    # ---------------------------------------------
    # Learning-rate scheduler
    # ---------------------------------------------

    scheduler.step(val_loss)

    # ---------------------------------------------
    # Store history
    # ---------------------------------------------

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    # ---------------------------------------------
    # Print metrics
    # ---------------------------------------------

    print(
        f"Train Loss : {train_loss:.4f}   "
        f"Train Acc : {train_acc:.2f}%"
    )

    print(
        f"Val   Loss : {val_loss:.4f}   "
        f"Val   Acc : {val_acc:.2f}%   "
        f"Val Macro-F1 : {val_macro_f1:.4f}"
    )

    # ---------------------------------------------
    # Save best model based on Macro-F1
    # ---------------------------------------------

    if val_macro_f1 > best_val_f1:

        best_val_f1 = val_macro_f1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            CHECKPOINT_PATH
        )

        print(
            f"Best model saved to {CHECKPOINT_PATH}! "
            f"(val macro-F1 = {val_macro_f1:.4f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOP_PATIENCE}"
        )

        if epochs_without_improvement >= EARLY_STOP_PATIENCE:

            print(
                f"\nNo improvement for "
                f"{EARLY_STOP_PATIENCE} epochs — "
                f"stopping early at epoch {epoch + 1}."
            )

            break


# =========================================================
# Restore best checkpoint
# =========================================================

if os.path.exists(CHECKPOINT_PATH):

    model.load_state_dict(
        torch.load(
            CHECKPOINT_PATH,
            map_location=device
        )
    )

    model.to(device)

    print(
        "\nBest model restored from:",
        CHECKPOINT_PATH
    )


print(
    "\nTraining complete."
)

print(
    f"Best validation Macro-F1: {best_val_f1:.4f}"
)

Epoch [1/30]


Train Loss : 28.4793   Train Acc : 38.78%
Val   Loss : 2.3459   Val   Acc : 62.44%   Val Macro-F1 : 0.5526
Best model saved to BerryNet_best_transfer.pth! (val macro-F1 = 0.5526)
Epoch [2/30]


Train Loss : 5.6986   Train Acc : 35.59%
Val   Loss : 1.0674   Val   Acc : 14.72%   Val Macro-F1 : 0.0855
No improvement: 1/7
Epoch [3/30]


Training:  64%|██████▍   | 16/25 [03:28<01:53, 12.65s/it]

## 5. Test-set evaluation + accuracy target check + class-balance / bias report

This is the section that actually answers "is it biased?" and "did it hit
90%?" — with numbers, not assumptions.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns


# =========================================================
# Class names
# =========================================================

# train_dataset is a Subset, so access the original
# ImageFolder dataset through .dataset
class_names = train_dataset.dataset.classes

print("Classes:", class_names)


# =========================================================
# Load best checkpoint
# =========================================================

model.load_state_dict(
    torch.load(
        CHECKPOINT_PATH,
        map_location=device
    )
)

model.to(device)
model.eval()


# =========================================================
# Test predictions
# =========================================================

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predicted = torch.argmax(
            outputs,
            dim=1
        )

        all_predictions.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )


# =========================================================
# Overall test metrics
# =========================================================

test_acc = (
    accuracy_score(
        all_labels,
        all_predictions
    ) * 100
)

weighted_f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)


print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Weighted F1  : {weighted_f1:.4f}")
print(f"Macro F1     : {macro_f1:.4f}")
print()


# =========================================================
# Classification report
# =========================================================

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        zero_division=0
    )
)


# =========================================================
# Confusion matrix
# =========================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(
    f"Confusion Matrix ({MODEL_TYPE})"
)

plt.tight_layout()
plt.show()


# =========================================================
# Accuracy target check
# =========================================================

TARGET_ACC = 90.0

print("=" * 60)

if test_acc >= TARGET_ACC:

    print(
        f"ACCURACY TARGET: PASS — "
        f"{test_acc:.2f}% >= {TARGET_ACC}%"
    )

else:

    print(
        f"ACCURACY TARGET: NOT MET — "
        f"{test_acc:.2f}% < {TARGET_ACC}%"
    )

    print(
        "Possible next steps:"
    )

    print(
        "  1. Compare custom BerryNet with transfer learning."
    )

    print(
        "  2. Increase EPOCHS or fine-tune more "
        "EfficientNet layers."
    )

    print(
        "  3. Review the weakest class and dataset quality."
    )


# =========================================================
# Per-class accuracy
# =========================================================

per_class_acc, overall_acc, _ = evaluate_per_class(
    model,
    test_loader,
    class_names,
    device
)


print("=" * 60)
print("CLASS-WISE ACCURACY")

for name, acc in per_class_acc.items():

    if acc is not None:
        print(
            f"  {name:22s}: {acc:.2f}%"
        )
    else:
        print(
            f"  {name:22s}: No test samples"
        )


# =========================================================
# Class accuracy gap
# =========================================================

valid_accs = [
    acc
    for acc in per_class_acc.values()
    if acc is not None
]

if len(valid_accs) >= 2:

    bias_gap = (
        max(valid_accs) -
        min(valid_accs)
    )

    worst_class = min(
        per_class_acc,
        key=lambda k:
        per_class_acc[k]
        if per_class_acc[k] is not None
        else float("inf")
    )

    print(
        f"\nClass accuracy gap "
        f"(best - worst): {bias_gap:.2f} points"
    )

    print(
        f"Lowest-performing class: {worst_class}"
    )

else:

    print(
        "\nClass accuracy gap cannot be calculated "
        "because fewer than two classes have test samples."
    )

## 5.1 Custom BerryNet vs. EfficientNet-B0 -- head-to-head comparison

This is the actual evidence for "why two models": both architectures are
trained here **the same way** (same data, same loss function, same number
of epochs, same optimizer family) so that any difference in the results
is attributable to the architecture itself, not to a training-setup
difference. Both are then evaluated on the identical held-out test set.

**This retrains both models from scratch and will take a while** -- run it
once you're happy with your main `MODEL_TYPE` config above, as a dedicated
comparison run, not as part of routine experimentation.

In [ ]:
import time

COMPARISON_EPOCHS = 15
COMPARISON_LOSS = nn.CrossEntropyLoss(label_smoothing=0.1)


def build_and_train_for_comparison(model_type, epochs=COMPARISON_EPOCHS):

    # Reproducibility
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Build model
    m = build_model(model_type, num_classes=num_classes).to(device)

    # Transfer learning setup
    if model_type == "transfer":
        # Freeze all layers
        for p in m.parameters():
            p.requires_grad = False

        # Train classifier
        for p in m.classifier.parameters():
            p.requires_grad = True

        # Fine-tune last 3 feature blocks
        for block in list(m.features.children())[-3:]:
            for p in block.parameters():
                p.requires_grad = True

        lr = 3e-4

    else:
        # Custom BerryNet
        lr = 1e-3

    # Optimizer
    opt = optim.Adam(
        [p for p in m.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=1e-4
    )

    # Learning-rate scheduler
    sched = optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=0.1,
        patience=3
    )

    best_f1 = -1.0
    ckpt_path = f"comparison_{model_type}.pth"

    start_time = time.time()

    # -------------------------
    # Training
    # -------------------------
    for epoch in range(epochs):

        m.train()

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            opt.zero_grad(set_to_none=True)

            outputs = m(images)

            loss = COMPARISON_LOSS(outputs, labels)

            loss.backward()
            opt.step()

        # -------------------------
        # Validation
        # -------------------------
        m.eval()

        val_correct = 0
        val_total = 0
        val_loss_sum = 0.0

        all_p = []
        all_l = []

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = m(images)

                loss = COMPARISON_LOSS(outputs, labels)

                val_loss_sum += loss.item()

                _, preds = torch.max(outputs, 1)

                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                all_p.extend(preds.cpu().numpy())
                all_l.extend(labels.cpu().numpy())

        val_loss = val_loss_sum / max(len(val_loader), 1)

        val_acc = 100.0 * val_correct / max(val_total, 1)

        val_f1 = f1_score(
            all_l,
            all_p,
            average="macro",
            zero_division=0
        )

        # Scheduler uses validation loss
        sched.step(val_loss)

        # Save best model based on validation Macro-F1
        if val_f1 > best_f1:

            best_f1 = val_f1

            torch.save(
                m.state_dict(),
                ckpt_path
            )

        print(
            f"  [{model_type}] "
            f"epoch {epoch + 1}/{epochs} "
            f"val_acc={val_acc:.2f}% "
            f"val_macro_f1={val_f1:.4f}"
        )

    # -------------------------
    # Training time
    # -------------------------
    training_time = time.time() - start_time

    # -------------------------
    # Load best model
    # -------------------------
    m.load_state_dict(
        torch.load(
            ckpt_path,
            map_location=device
        )
    )

    m.to(device)
    m.eval()

    # -------------------------
    # Test evaluation
    # -------------------------
    per_class_acc, test_acc, test_macro_f1 = evaluate_per_class(
        m,
        test_loader,
        class_names,
        device
    )

    # Number of parameters
    n_params = sum(
        p.numel()
        for p in m.parameters()
    )

    return {
        "model_type": model_type,
        "test_accuracy": test_acc,
        "macro_f1": test_macro_f1,
        "per_class_accuracy": per_class_acc,
        "n_params": n_params,
        "training_time_sec": training_time,
    }


# ==========================================
# Run comparison
# ==========================================

comparison_results = []

for mt in ["custom", "transfer"]:

    print(
        f"\n{'=' * 60}\n"
        f"Training {mt} for comparison\n"
        f"{'=' * 60}"
    )

    result = build_and_train_for_comparison(mt)

    comparison_results.append(result)

In [ ]:
import pandas as pd

# ==========================================
# Create comparison table
# ==========================================

rows = []

for r in comparison_results:

    row = {
        "Model": (
            "Custom BerryNet"
            if r["model_type"] == "custom"
            else "EfficientNet-B0 (transfer)"
        ),
        "Test accuracy (%)": round(r["test_accuracy"], 2),
        "Macro F1": round(r["macro_f1"], 4),
        "Parameters": f"{r['n_params']:,}",
        "Training time (s)": round(r["training_time_sec"], 1),
    }

    # Add per-class accuracy
    for cls, acc in r["per_class_accuracy"].items():

        row[f"{cls} acc (%)"] = (
            round(acc, 2)
            if acc is not None
            else None
        )

    rows.append(row)


comparison_df = pd.DataFrame(rows)

print(comparison_df.to_string(index=False))


# ==========================================
# Plot comparison
# ==========================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4)
)

labels = [
    r["Model"]
    for r in rows
]


# Test Accuracy
axes[0].bar(
    labels,
    [r["Test accuracy (%)"] for r in rows]
)

axes[0].set_title("Test Accuracy (%)")
axes[0].set_ylim(0, 100)
axes[0].set_ylabel("Accuracy (%)")


# Macro F1
axes[1].bar(
    labels,
    [r["Macro F1"] for r in rows]
)

axes[1].set_title("Macro F1")
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Macro F1")


# Format x-axis
for ax in axes:
    ax.tick_params(
        axis="x",
        rotation=15
    )

plt.tight_layout()
plt.show()

## 6. Grad-CAM

Works for either `MODEL_TYPE` — the hook point is selected automatically via
`get_gradcam_target_layer()` defined in Section 3.

In [ ]:
import cv2


class GradCAM:

    def __init__(self, model, target_layer):

        self.model = model
        self.gradients = None
        self.activations = None

        # Store hook handles so they can be removed later
        self.forward_handle = target_layer.register_forward_hook(
            self._save_activation
        )

        self.backward_handle = target_layer.register_full_backward_hook(
            self._save_gradient
        )

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):

        if grad_output[0] is not None:
            self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):

        self.model.eval()

        # Clear previous values
        self.activations = None
        self.gradients = None

        # Forward pass
        output = self.model(input_tensor)

        # Use predicted class if class is not specified
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        # Backward pass
        self.model.zero_grad(set_to_none=True)

        score = output[0, class_idx]

        score.backward()

        # Safety checks
        if self.activations is None:
            raise RuntimeError("Grad-CAM activations were not captured.")

        if self.gradients is None:
            raise RuntimeError("Grad-CAM gradients were not captured.")

        # Global average pooling of gradients
        weights = self.gradients.mean(
            dim=(2, 3),
            keepdim=True
        )

        # Weighted combination of activation maps
        cam = (weights * self.activations).sum(
            dim=1,
            keepdim=True
        )

        # Keep only positive activations
        cam = torch.relu(cam)

        # Convert to NumPy
        cam = cam.squeeze().cpu().numpy()

        # Resize CAM to input image size
        cam = cv2.resize(
            cam,
            (
                input_tensor.shape[-1],
                input_tensor.shape[-2]
            )
        )

        # Normalize between 0 and 1
        cam = cam - cam.min()

        cam = cam / (cam.max() + 1e-8)

        return cam, class_idx

    def remove_hooks(self):
        """Remove Grad-CAM hooks when they are no longer needed."""
        self.forward_handle.remove()
        self.backward_handle.remove()


def overlay_heatmap(image_pil, cam, alpha=0.45):

    image_np = np.array(
        image_pil.convert("RGB")
    )

    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam),
        cv2.COLORMAP_JET
    )

    heatmap = cv2.cvtColor(
        heatmap,
        cv2.COLOR_BGR2RGB
    )

    overlay = (
        heatmap * alpha
        + image_np * (1 - alpha)
    ).astype(np.uint8)

    return overlay


# ==========================================
# Initialize Grad-CAM
# ==========================================

gradcam_target = get_gradcam_target_layer(
    model,
    MODEL_TYPE
)

gradcam = GradCAM(
    model,
    target_layer=gradcam_target
)

print(
    f"Grad-CAM ready for model_type='{MODEL_TYPE}'"
)

## 7. Severity scoring (lesion-area segmentation)

Same HSV-based approach as before — sanity-check on your own images before
trusting the numbers, using the preview cell below.

In [ ]:
def segment_leaf_and_lesion(image_pil, min_blob_area_frac=0.002):
    """
    Segment the leaf and estimate lesion area using HSV/LAB color information.

    The method:
    1. Creates a leaf mask from saturation/value.
    2. Keeps the largest connected leaf region.
    3. Excludes very bright glare and very dark shadow regions.
    4. Detects non-healthy-looking color regions in LAB space.
    5. Removes very small connected components.
    6. Calculates lesion percentage and severity.
    """

    # ------------------------------------------
    # Convert image
    # ------------------------------------------
    image_np = np.array(
        image_pil.convert("RGB")
    )

    hsv = cv2.cvtColor(
        image_np,
        cv2.COLOR_RGB2HSV
    )

    lab = cv2.cvtColor(
        image_np,
        cv2.COLOR_RGB2LAB
    )

    h, s, v = cv2.split(hsv)
    l, a, b = cv2.split(lab)

    # ------------------------------------------
    # Leaf segmentation
    # ------------------------------------------
    leaf_mask = (
        (s > 20) &
        (v > 25)
    ).astype(np.uint8) * 255

    kernel_small = np.ones(
        (5, 5),
        np.uint8
    )

    kernel_med = np.ones(
        (9, 9),
        np.uint8
    )

    # Fill small gaps
    leaf_mask = cv2.morphologyEx(
        leaf_mask,
        cv2.MORPH_CLOSE,
        kernel_med
    )

    # Remove small noise
    leaf_mask = cv2.morphologyEx(
        leaf_mask,
        cv2.MORPH_OPEN,
        kernel_small
    )

    # ------------------------------------------
    # Keep largest connected component
    # ------------------------------------------
    num_labels, labels_im, stats, _ = (
        cv2.connectedComponentsWithStats(
            leaf_mask,
            connectivity=8
        )
    )

    if num_labels > 1:

        largest_label = (
            1 +
            np.argmax(
                stats[1:, cv2.CC_STAT_AREA]
            )
        )

        leaf_mask = np.where(
            labels_im == largest_label,
            255,
            0
        ).astype(np.uint8)

        # Fill the selected leaf contour
        contours, _ = cv2.findContours(
            leaf_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        leaf_mask = np.zeros_like(
            leaf_mask
        )

        cv2.drawContours(
            leaf_mask,
            contours,
            -1,
            255,
            thickness=cv2.FILLED
        )

    # ------------------------------------------
    # Exclude glare and strong shadows
    # ------------------------------------------
    glare_mask = (
        (v > 235) &
        (s < 40)
    )

    shadow_mask = (
        v < 35
    )

    excluded = (
        glare_mask |
        shadow_mask
    )

    # ------------------------------------------
    # LAB-based lesion candidate
    # ------------------------------------------
    a_centered = (
        a.astype(np.int16) - 128
    )

    b_centered = (
        b.astype(np.int16) - 128
    )

    # Approximate healthy green region
    healthy_like = (
        (a_centered < 2) &
        (b_centered < 35)
    )

    lesion_candidate = ~healthy_like

    # Only consider pixels inside the leaf
    lesion_mask = (
        (leaf_mask > 0) &
        lesion_candidate &
        (~excluded)
    )

    lesion_mask = (
        lesion_mask.astype(np.uint8) * 255
    )

    # Remove small isolated pixels
    lesion_mask = cv2.morphologyEx(
        lesion_mask,
        cv2.MORPH_OPEN,
        kernel_small
    )

    # ------------------------------------------
    # Calculate leaf area
    # ------------------------------------------
    leaf_area_px = int(
        np.sum(leaf_mask > 0)
    )

    if leaf_area_px == 0:
        return (
            leaf_mask,
            np.zeros_like(leaf_mask),
            0.0,
            "Healthy / negligible"
        )

    # ------------------------------------------
    # Connected-component filtering
    # ------------------------------------------
    min_blob_px = max(
        15,
        int(leaf_area_px * min_blob_area_frac)
    )

    num_labels, labels_im, stats, _ = (
        cv2.connectedComponentsWithStats(
            lesion_mask,
            connectivity=8
        )
    )

    cleaned = np.zeros_like(
        lesion_mask
    )

    for lbl in range(1, num_labels):

        area = stats[
            lbl,
            cv2.CC_STAT_AREA
        ]

        w_box = stats[
            lbl,
            cv2.CC_STAT_WIDTH
        ]

        h_box = stats[
            lbl,
            cv2.CC_STAT_HEIGHT
        ]

        aspect_ratio = (
            max(w_box, h_box) /
            max(1, min(w_box, h_box))
        )

        # Remove tiny components and extremely
        # elongated noise regions
        if (
            area >= min_blob_px and
            aspect_ratio < 6
        ):
            cleaned[
                labels_im == lbl
            ] = 255

    lesion_mask = cleaned

    # ------------------------------------------
    # Severity calculation
    # ------------------------------------------
    lesion_area = int(
        np.sum(lesion_mask > 0)
    )

    severity_pct = (
        100.0 * lesion_area / leaf_area_px
    )

    # Preserve your original thresholds
    if severity_pct < 2:
        severity_label = "Healthy / negligible"

    elif severity_pct < 10:
        severity_label = "Mild"

    elif severity_pct < 25:
        severity_label = "Moderate"

    else:
        severity_label = "Severe"

    return (
        leaf_mask,
        lesion_mask,
        severity_pct,
        severity_label
    )


# ==========================================
# Segmentation preview
# ==========================================

def preview_segmentation(image_path):

    from PIL import Image

    image_pil = (
        Image.open(image_path)
        .convert("RGB")
        .resize((IMG_SIZE, IMG_SIZE))
    )

    (
        leaf_mask,
        lesion_mask,
        severity_pct,
        severity_label
    ) = segment_leaf_and_lesion(
        image_pil
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 4)
    )

    axes[0].imshow(image_pil)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(
        leaf_mask,
        cmap="gray"
    )
    axes[1].set_title("Leaf mask")
    axes[1].axis("off")

    axes[2].imshow(
        lesion_mask,
        cmap="gray"
    )
    axes[2].set_title(
        f"Lesion mask\n"
        f"{severity_label} ({severity_pct:.1f}%)"
    )
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()


# Example:
# preview_segmentation(test_base.samples[0][0])

## 8. Combined output — class + confidence + severity + Grad-CAM

## 8.1 Feeding and treatment recommendation

This turns the classification + severity output into a concrete action for
a sericulture farmer: **is this leaf safe to feed to silkworms**, and what
general treatment direction fits the diagnosed disease and severity level.

**Important scope note:** this is a rule-based mapping from (disease class,
severity level) to general guidance, not a validated agronomic/veterinary
recommendation. It should be presented to end users as a *starting point
for a decision*, with an explicit prompt to confirm anything specific
(chemical names, dosages, timing) with a local agricultural extension
officer or sericulture specialist. Silkworms are highly sensitive to leaf
quality and to any chemical residue on leaves, so incorrect feeding advice
carries a real economic (and animal welfare) cost -- this feature should be
framed conservatively, not as authoritative treatment instructions.

In [ ]:
# Get class names from the original dataset
class_names = train_dataset.dataset.classes

print("Class names:")
for i, name in enumerate(class_names):
    print(i, "->", name)

In [ ]:
# Detect healthy class names
HEALTHY_CLASS_NAMES = [
    name for name in class_names
    if "healthy" in name.lower()
]

print("Healthy class names:", HEALTHY_CLASS_NAMES)

In [ ]:
def get_recommendation(
    disease_class,
    severity_label,
    healthy_class_names=None
):

    if healthy_class_names is None:
        healthy_class_names = HEALTHY_CLASS_NAMES

    # Normalize severity label
    severity = str(severity_label).strip().lower()

    # Check healthy class
    is_healthy = disease_class in healthy_class_names

    # Healthy
    if is_healthy or severity.startswith("healthy"):

        return {
            "feeding_advice":
                "The leaf is classified as healthy and can generally "
                "be used for feeding silkworms.",

            "treatment_advice":
                "No disease treatment is indicated. Continue regular "
                "monitoring and normal plant care.",

            "urgency": "None"
        }

    # Mild
    elif severity == "mild":

        return {
            "feeding_advice":
                "Remove visibly affected leaves and prioritize healthy "
                "leaves for feeding.",

            "treatment_advice":
                f"Early-stage {disease_class} was detected. "
                "Monitor the plant regularly and remove affected "
                "plant material.",

            "urgency": "Low"
        }

    # Moderate
    elif severity == "moderate":

        return {
            "feeding_advice":
                "Avoid feeding heavily affected leaves. Use healthy "
                "leaves wherever possible.",

            "treatment_advice":
                f"Moderate {disease_class} symptoms were detected. "
                "Remove severely affected leaves and monitor nearby plants.",

            "urgency": "Medium"
        }

    # Severe
    elif severity == "severe":

        return {
            "feeding_advice":
                "Do not use severely affected leaves for silkworm feeding. "
                "Use healthy leaves instead.",

            "treatment_advice":
                f"Severe {disease_class} symptoms were detected. "
                "Remove severely affected plant material and seek "
                "appropriate agricultural guidance.",

            "urgency": "High"
        }

    # Unknown
    else:

        return {
            "feeding_advice":
                "Use healthy leaves for feeding until the condition "
                "is confirmed.",

            "treatment_advice":
                "The severity could not be determined. Monitor the "
                "plant and verify the diagnosis.",

            "urgency": "Unknown"
        }

## 8.2 Optional: LLM-phrased recommendation (hybrid approach)

The rule-based `get_recommendation()` above remains the **source of truth**
for the actual decision -- whether to feed, and the urgency level. This
section optionally sends that same decision to an LLM (Claude) purely to
**rephrase it more naturally** for a farmer-facing app.

**Guardrail, by design:** the LLM is explicitly instructed not to introduce
any new facts, chemical names, or dosages beyond what the rule-based
recommendation already states -- it can only reword, not add content. This
keeps the actual agricultural decision deterministic and auditable while
still allowing more natural, elaborated phrasing where useful.

**This is fully optional.** If you don't provide an API key, the notebook
falls back to the plain rule-based text automatically -- nothing breaks.

In [ ]:
import os
import getpass
import json
import requests

# Optional: provide an Anthropic API key to enable natural-language
# rephrasing of the recommendation.
# Leave blank to skip LLM phrasing.
ANTHROPIC_API_KEY = os.environ.get(
    "ANTHROPIC_API_KEY",
    ""
)

if not ANTHROPIC_API_KEY:

    try:
        ANTHROPIC_API_KEY = getpass.getpass(
            "Enter Anthropic API key to enable LLM phrasing "
            "(leave blank to skip): "
        ).strip()

    except Exception:
        ANTHROPIC_API_KEY = ""

USE_LLM_PHRASING = bool(
    ANTHROPIC_API_KEY
)

print(
    "LLM phrasing enabled:",
    USE_LLM_PHRASING
)


def phrase_recommendation_naturally(
    disease_class,
    severity_label,
    severity_pct,
    recommendation,
    api_key=None
):
    """
    Rephrase the existing rule-based recommendation.

    The rule-based recommendation remains the source of truth.
    The LLM is only used to make the wording more natural.

    If the API key is missing or the API request fails,
    the original recommendation is returned unchanged.
    """

    api_key = api_key or ANTHROPIC_API_KEY

    # No API key -> use rule-based recommendation
    if not api_key:
        return recommendation

    system_prompt = (
        "You are rephrasing agricultural advice for a "
        "farmer-facing app. "

        "You MUST NOT add any new facts, chemical names, "
        "dosages, or recommendations beyond what is given "
        "to you. "

        "Only rephrase the given feeding advice and treatment "
        "direction to sound natural, clear, and appropriately "
        "urgent for a farmer with no technical background. "

        "Keep each to 2-3 short sentences. "

        "Respond ONLY as raw JSON with exactly these keys: "
        '{"feeding_advice": "...", '
        '"treatment_advice": "..."}'
    )

    user_prompt = (
        f"Disease: {disease_class}\n"
        f"Severity: {severity_label} "
        f"({severity_pct:.1f}% of leaf area affected)\n"
        f"Urgency: {recommendation['urgency']}\n"
        f"Feeding advice (facts - do not change the meaning): "
        f"{recommendation['feeding_advice']}\n"
        f"Treatment direction (facts - do not change the meaning): "
        f"{recommendation['treatment_advice']}\n"
    )

    try:

        response = requests.post(
            "https://api.anthropic.com/v1/messages",

            headers={
                "x-api-key": api_key,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },

            json={
                "model": "claude-haiku-4-5-20251001",
                "max_tokens": 300,
                "system": system_prompt,
                "messages": [
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
            },

            timeout=15,
        )

        response.raise_for_status()

        data = response.json()

        text = "".join(
            block.get("text", "")
            for block in data.get("content", [])
        )

        # Remove possible markdown code fences
        text_clean = text.strip()

        if text_clean.startswith("```"):
            text_clean = text_clean.strip("`").strip()

            if text_clean.lower().startswith("json"):
                text_clean = text_clean[4:].strip()

        parsed = json.loads(text_clean)

        # Keep original recommendation as the base
        natural = dict(recommendation)

        natural["feeding_advice"] = parsed.get(
            "feeding_advice",
            recommendation["feeding_advice"]
        )

        natural["treatment_advice"] = parsed.get(
            "treatment_advice",
            recommendation["treatment_advice"]
        )

        natural["phrasing"] = "llm"

        return natural

    except Exception as e:

        print(
            "(LLM phrasing failed, falling back to "
            f"rule-based text: {e})"
        )

        return recommendation

In [ ]:
from PIL import Image


# ==========================================
# Detect healthy class names
# ==========================================

HEALTHY_CLASS_NAMES = [
    c for c in class_names
    if "healthy" in c.lower()
]

print(
    "Detected healthy class name(s):",
    HEALTHY_CLASS_NAMES
)


# ==========================================
# Diagnosis function
# ==========================================

def diagnose(
    image_path,
    model=model,
    transform=test_transform,
    class_names=class_names,
    device=device,
    gradcam=gradcam,
    healthy_class_names=HEALTHY_CLASS_NAMES,
    healthy_confidence_threshold=0.6,
    use_llm_phrasing=None
):

    # --------------------------------------
    # Decide whether LLM phrasing is used
    # --------------------------------------

    if use_llm_phrasing is None:
        use_llm_phrasing = USE_LLM_PHRASING


    # --------------------------------------
    # Load image
    # --------------------------------------

    image_pil = (
        Image.open(image_path)
        .convert("RGB")
    )

    image_resized = image_pil.resize(
        (IMG_SIZE, IMG_SIZE)
    )


    # --------------------------------------
    # Prepare tensor
    # --------------------------------------

    image_tensor = (
        transform(image_pil)
        .unsqueeze(0)
        .to(device)
    )


    # --------------------------------------
    # Classification
    # --------------------------------------

    model.eval()

    with torch.no_grad():

        output = model(image_tensor)

        probabilities = torch.softmax(
            output,
            dim=1
        )

        confidence, prediction = torch.max(
            probabilities,
            dim=1
        )

    pred_class = class_names[
        prediction.item()
    ]


    # --------------------------------------
    # Grad-CAM
    # --------------------------------------

    cam, cam_class_idx = gradcam.generate(
        image_tensor.clone().requires_grad_(True),
        class_idx=prediction.item()
    )

    overlay = overlay_heatmap(
        image_resized,
        cam
    )


    # --------------------------------------
    # Leaf / lesion segmentation
    # --------------------------------------

    (
        leaf_mask,
        lesion_mask,
        severity_pct,
        severity_label
    ) = segment_leaf_and_lesion(
        image_resized
    )


    # --------------------------------------
    # Healthy-class safety net
    # --------------------------------------

    overridden = False

    if (
        pred_class in healthy_class_names
        and confidence.item() > healthy_confidence_threshold
    ):

        if severity_pct > 5.0:
            overridden = True

        # Preserve the original safety-net behavior
        severity_pct = min(
            severity_pct,
            5.0
        )

        severity_label = (
            "Healthy / negligible"
        )


    # --------------------------------------
    # Rule-based recommendation
    # --------------------------------------

    recommendation = get_recommendation(
        pred_class,
        severity_label,
        healthy_class_names
    )


    # --------------------------------------
    # Optional LLM rephrasing
    # --------------------------------------

    if use_llm_phrasing:

        recommendation = (
            phrase_recommendation_naturally(
                pred_class,
                severity_label,
                severity_pct,
                recommendation
            )
        )


    # ======================================
    # Visualization
    # ======================================

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(14, 5)
    )


    # Original image
    axes[0].imshow(
        image_resized
    )

    axes[0].set_title(
        "Input leaf"
    )

    axes[0].axis("off")


    # Grad-CAM
    axes[1].imshow(
        overlay
    )

    axes[1].set_title(
        f"Grad-CAM: {pred_class}"
    )

    axes[1].axis("off")


    # Lesion mask
    axes[2].imshow(
        lesion_mask,
        cmap="Reds"
    )

    axes[2].set_title(
        f"Lesion area: {severity_label}"
    )

    axes[2].axis("off")


    plt.tight_layout()
    plt.show()


    # ======================================
    # Console output
    # ======================================

    print(
        f"Prediction : {pred_class}"
    )

    print(
        f"Confidence : "
        f"{confidence.item() * 100:.2f}%"
    )


    print(
        "All class probabilities:"
    )

    for name, p in zip(
        class_names,
        probabilities[0].cpu().numpy()
    ):

        print(
            f"  {name:22s}: "
            f"{p * 100:.2f}%"
        )


    print()

    print(
        f"Estimated severity : "
        f"{severity_label} "
        f"({severity_pct:.1f}% of leaf area affected)"
    )


    if overridden:

        print(
            "(Severity capped: classifier predicted "
            "healthy with high confidence, while the "
            "segmentation produced a higher raw severity "
            "estimate.)"
        )


    print()


    phrasing_tag = (
        " (LLM-phrased)"
        if recommendation.get("phrasing") == "llm"
        else " (rule-based)"
    )


    print(
        f"Silkworm feeding advice"
        f"{phrasing_tag} "
        f"[{recommendation['urgency']} urgency]:"
    )

    print(
        f"  {recommendation['feeding_advice']}"
    )


    print(
        f"Treatment direction"
        f"{phrasing_tag}:"
    )

    print(
        f"  {recommendation['treatment_advice']}"
    )


    # ======================================
    # Return structured result
    # ======================================

    return {
        "prediction": pred_class,

        "confidence": confidence.item(),

        "probabilities": dict(
            zip(
                class_names,
                probabilities[0].cpu().tolist()
            )
        ),

        "severity_pct": severity_pct,

        "severity_label": severity_label,

        "severity_overridden": overridden,

        "recommendation": recommendation,
    }


# ==========================================
# Example usage
# ==========================================

# result = diagnose(test_base.samples[0][0])

# Force LLM phrasing:
# result = diagnose(
#     test_base.samples[0][0],
#     use_llm_phrasing=True
# )

# Force rule-based phrasing:
# result = diagnose(
#     test_base.samples[0][0],
#     use_llm_phrasing=False
# )

## 9. Continuous learning demonstration

**Read this before running:** this section is a *demonstration* of continual
learning on your dataset, done honestly — it does not pretend to be a
production streaming system, because you don't have one (your data comes
from a single Kaggle download, not a live pipeline). What it does show,
with real numbers, is the actual problem continual learning solves and a
working fix:

1. Train a model on only 2 of your 3 classes ("Phase A" — simulating "this is
   what the model knew before deployment").
2. **Naive update**: fine-tune that model on the 3rd class only ("Phase B" —
   simulating "new data arrived"). This is expected to *hurt* accuracy on
   the original 2 classes — this is catastrophic forgetting, the actual
   problem continual learning exists to solve.
3. **Replay-buffer update**: repeat the Phase B fine-tune, but mix in a
   small held-back sample of Phase A images alongside the new class. This
   should retain most of the original 2 classes' accuracy while still
   learning the new class.
4. Compare all three per-class, in a table — the result you'd actually put
   in your report/viva as evidence the technique works.

This uses your best-performing model type/config from above as the starting
architecture (fresh weights, not the already-3-class-trained checkpoint).

### Note on novelty given Nobel et al. (2026)

Nobel et al.'s paper — the closest prior work to this notebook's
architecture — does classification, severity is not addressed at all, and
their explainability is SHAP (leaf species) + Grad-CAM (disease detection).
**They do not include any continuous/incremental learning experiment** —
their future-work section mentions drone/IoT deployment, not model updating
over time. The Phase A / naive / replay-buffer comparison below is therefore
still a legitimate point of difference from the published literature this
notebook is built on, provided it's reported with the same honest framing
used throughout this notebook: a controlled simulation on a static dataset,
not a deployed streaming system.

In [ ]:
import copy

# ============================================================
# Phase A / Phase B class configuration
# ============================================================

# By default, the last class is treated as the new class
# introduced during Phase B.
PHASE_B_CLASS_IDX = num_classes - 1

# All remaining classes are considered known during Phase A.
PHASE_A_CLASS_IDXS = [
    i for i in range(num_classes)
    if i != PHASE_B_CLASS_IDX
]


print(
    "Phase A (known) classes:",
    [class_names[i] for i in PHASE_A_CLASS_IDXS]
)

print(
    "Phase B (new) class    :",
    class_names[PHASE_B_CLASS_IDX]
)


# ============================================================
# Split training indices into Phase A and Phase B
# ============================================================

phase_a_train_idx = [
    i for i in train_idx
    if base_dataset.targets[i] in PHASE_A_CLASS_IDXS
]

phase_b_train_idx = [
    i for i in train_idx
    if base_dataset.targets[i] == PHASE_B_CLASS_IDX
]


# ============================================================
# Replay buffer
# ============================================================

# Keep a portion of Phase A images so that old knowledge
# can be replayed during Phase B training.
REPLAY_FRACTION = 0.25

rng = np.random.RandomState(SEED)


if len(phase_a_train_idx) > 0:

    replay_size = min(
        len(phase_a_train_idx),
        max(
            20,
            int(
                len(phase_a_train_idx)
                * REPLAY_FRACTION
            )
        )
    )

    replay_idx = list(
        rng.choice(
            phase_a_train_idx,
            size=replay_size,
            replace=False
        )
    )

else:
    replay_size = 0
    replay_idx = []


# ============================================================
# DataLoaders
# ============================================================

phase_a_loader = DataLoader(
    Subset(train_base, phase_a_train_idx),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)


phase_b_only_loader = DataLoader(
    Subset(train_base, phase_b_train_idx),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)


phase_b_plus_replay_loader = DataLoader(
    Subset(
        train_base,
        phase_b_train_idx + replay_idx
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)


# ============================================================
# Summary
# ============================================================

print(
    f"Phase A training images: "
    f"{len(phase_a_train_idx)}"
)

print(
    f"Phase B training images: "
    f"{len(phase_b_train_idx)}"
)

print(
    f"Replay buffer size     : "
    f"{len(replay_idx)}"
)

In [ ]:
CL_EPOCHS_PHASE_A = 8
CL_EPOCHS_UPDATE = 5


def simple_train(model, loader, epochs, lr=1e-4):

    opt = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    crit = nn.CrossEntropyLoss()

    for ep in range(epochs):

        model.train()
        running_loss = 0.0

        for images, labels in tqdm(
            loader,
            desc=f"epoch {ep + 1}/{epochs}",
            leave=False
        ):

            images = images.to(device)
            labels = labels.to(device)

            opt.zero_grad(set_to_none=True)

            out = model(images)

            loss = crit(
                out,
                labels
            )

            loss.backward()
            opt.step()

            running_loss += loss.item()

        avg_loss = (
            running_loss / len(loader)
            if len(loader) > 0
            else 0.0
        )

        print(
            f"Epoch {ep + 1}/{epochs} "
            f"- Loss: {avg_loss:.4f}"
        )

    return model


# ============================================================
# Step 1: Phase A baseline model
# ============================================================
# The model is trained only on the known Phase A classes.
# The Phase B class has not been seen during this training.

phase_a_model = build_model(
    MODEL_TYPE,
    num_classes=num_classes
).to(device)


phase_a_model = simple_train(
    phase_a_model,
    phase_a_loader,
    CL_EPOCHS_PHASE_A
)


# ============================================================
# Evaluate Phase A model on the complete test set
# ============================================================

baseline_per_class, baseline_overall, _ = evaluate_per_class(
    phase_a_model,
    test_loader,
    class_names,
    device
)


print(
    "\nPhase A baseline "
    "(before seeing new class) "
    "— per-class test accuracy:"
)

for name, acc in baseline_per_class.items():

    if acc is not None:
        print(
            f"  {name:22s}: {acc:.2f}%"
        )
    else:
        print(
            f"  {name:22s}: N/A"
        )

In [ ]:
# ============================================================
# Step 2: Naive update — new class ONLY
# ============================================================
# No replay data is used here.
# This intentionally demonstrates catastrophic forgetting.

naive_model = copy.deepcopy(
    phase_a_model
)

naive_model = simple_train(
    naive_model,
    phase_b_only_loader,
    CL_EPOCHS_UPDATE,
    lr=5e-5
)


# ============================================================
# Evaluate naive continual-learning model
# ============================================================

naive_per_class, naive_overall, _ = evaluate_per_class(
    naive_model,
    test_loader,
    class_names,
    device
)


print(
    "\nNaive continual update "
    "(no replay) — per-class test accuracy:"
)

for name, acc in naive_per_class.items():

    if acc is not None:
        print(
            f"  {name:22s}: {acc:.2f}%"
        )
    else:
        print(
            f"  {name:22s}: N/A"
        )


print(
    f"\nNaive overall test accuracy: "
    f"{naive_overall:.2f}%"
)

In [ ]:
# ============================================================
# Step 3: Replay-buffer update
# ============================================================
# Fine-tune using:
#   1. The new Phase B class
#   2. A replay buffer containing some Phase A samples
#
# The replay samples help preserve knowledge of the old classes.

replay_model = copy.deepcopy(
    phase_a_model
)

replay_model = simple_train(
    replay_model,
    phase_b_plus_replay_loader,
    CL_EPOCHS_UPDATE,
    lr=5e-5
)


# ============================================================
# Evaluate replay model
# ============================================================

replay_per_class, replay_overall, _ = evaluate_per_class(
    replay_model,
    test_loader,
    class_names,
    device
)


print(
    "\nReplay-buffer continual update "
    "— per-class test accuracy:"
)

for name, acc in replay_per_class.items():

    if acc is not None:
        print(
            f"  {name:22s}: {acc:.2f}%"
        )
    else:
        print(
            f"  {name:22s}: N/A"
        )


print(
    f"\nReplay overall test accuracy: "
    f"{replay_overall:.2f}%"
)

In [ ]:
import pandas as pd
import numpy as np


# ============================================================
# Compare the three continual-learning stages
# ============================================================

comparison = pd.DataFrame({
    "Phase A baseline (before new class)": baseline_per_class,
    "Naive update (no replay)": naive_per_class,
    "Replay-buffer update": replay_per_class,
}).T


# Add overall test accuracy
comparison["Overall"] = [
    baseline_overall,
    naive_overall,
    replay_overall
]


print(
    comparison.round(2)
)


# ============================================================
# Calculate forgetting on OLD classes
# ============================================================

old_classes = [
    class_names[i]
    for i in PHASE_A_CLASS_IDXS
]


# Keep only classes for which all three accuracies are available
valid_old_classes = [
    c for c in old_classes
    if (
        baseline_per_class.get(c) is not None
        and naive_per_class.get(c) is not None
        and replay_per_class.get(c) is not None
    )
]


if valid_old_classes:

    naive_drops = [
        baseline_per_class[c] - naive_per_class[c]
        for c in valid_old_classes
    ]

    replay_drops = [
        baseline_per_class[c] - replay_per_class[c]
        for c in valid_old_classes
    ]

    naive_drop = np.mean(
        naive_drops
    )

    replay_drop = np.mean(
        replay_drops
    )


    print()
    print(
        "Average accuracy drop on OLD classes "
        f"— naive update  : {naive_drop:.2f} points"
    )

    print(
        "Average accuracy drop on OLD classes "
        f"— replay update : {replay_drop:.2f} points"
    )


    # ========================================================
    # Compare forgetting
    # ========================================================

    if replay_drop < naive_drop:

        print(
            "\nReplay buffer produced a smaller "
            "average accuracy drop on the old classes."
        )

    elif replay_drop > naive_drop:

        print(
            "\nReplay buffer produced a larger "
            "average accuracy drop on the old classes "
            "in this experiment."
        )

    else:

        print(
            "\nReplay and naive updates produced the "
            "same average accuracy drop on the old classes."
        )


else:

    print(
        "\nCould not calculate old-class accuracy drop "
        "because one or more class accuracies were unavailable."
    )

**What to report from this section:** the comparison table above, plus the
average old-class accuracy drop for naive vs. replay. That's your concrete
evidence for a continuous-learning claim — a model that keeps most of its
old-class performance while learning a new class, versus one that doesn't.

**What this section does *not* claim:** this is not a live/streaming
deployment, it doesn't use EWC or other more advanced continual-learning
methods, and the "phases" are a simulation on already-collected data. Be
upfront about that framing in your report — it's still a legitimate and
correctly-executed continual learning experiment, just scoped to what's
actually demonstrable from a static dataset.

## Summary of what this notebook gives you

- **Bias**: Section 5 prints a per-class accuracy report and an automatic bias-gap flag — check this after every training run, not just the overall accuracy number.
- **Accuracy target**: Section 5 also auto-checks against 90% and tells you the next lever to pull if you're short. `MODEL_TYPE = "transfer"` is your strongest lever.
- **Continuous learning**: Section 9, honestly scoped as a simulated two-phase experiment with a real naive-vs-replay comparison.
- **Severity + Grad-CAM**: Sections 6–8, unchanged from before.

Run the whole notebook once with `MODEL_TYPE = "transfer"`, check the Section 5 report, and come back with the numbers — I'll tell you plainly whether they're good enough or what to adjust next.

In [ ]:
def simple_diagnose(image_path):
    model.eval()

    # Load image
    image = Image.open(image_path).convert("RGB")

    # Apply the same transform used during testing
    input_tensor = test_transform(image).unsqueeze(0).to(device)

    # Prediction
    with torch.no_grad():
        output = model(input_tensor)

        probabilities = torch.softmax(output, dim=1)
        confidence, predicted_idx = torch.max(
            probabilities,
            dim=1
        )

    predicted_idx = predicted_idx.item()
    confidence = confidence.item()

    # Get class name
    predicted_class = class_names[predicted_idx]

    # Simple severity estimate based on confidence
    if "healthy" in predicted_class.lower():
        severity_label = "Healthy"
    elif confidence >= 0.80:
        severity_label = "Severe"
    elif confidence >= 0.60:
        severity_label = "Moderate"
    else:
        severity_label = "Mild"

    # Recommendation
    recommendation = get_recommendation(
        disease_class=predicted_class,
        severity_label=severity_label
    )

    result = {
        "predicted_class": predicted_class,
        "confidence": confidence,
        "confidence_percentage": confidence * 100,
        "severity": severity_label,
        "feeding_advice": recommendation["feeding_advice"],
        "treatment_advice": recommendation["treatment_advice"],
        "urgency": recommendation["urgency"]
    }

    return result

In [ ]:
# ============================================================
# Run BerryNet diagnosis
# ============================================================

image_path = "/content/test_image.jpg"

result = diagnose(image_path)


print("====================================")
print("        BERRYNET DIAGNOSIS")
print("====================================")


print(
    "Predicted Class :",
    result["prediction"]
)

print(
    "Confidence      :",
    f"{result['confidence'] * 100:.2f}%"
)

print(
    "Severity        :",
    result["severity_label"]
)

print(
    "Severity Area   :",
    f"{result['severity_pct']:.2f}%"
)


recommendation = result["recommendation"]


print(
    "Urgency         :",
    recommendation["urgency"]
)


print("\nFeeding Advice:")
print(
    recommendation["feeding_advice"]
)


print("\nTreatment Advice:")
print(
    recommendation["treatment_advice"]
)


print("\n====================================")